<a href="https://www.kaggle.com/code/joshuanguyenolson/home-data-for-ml-course" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Home Prices: Random Forest Regression
**Updated:** 2026-06-09  
**Goal:** Predict residential home sale prices in Ames, Iowa  
**Metric:** RMSLE (Root Mean Squared Log Error)  
**Approach:** Feature engineering (imputation, encoding, skew correction) + RandomForestRegressor

## 1. Setup and Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics         import mean_squared_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 2. Data Loading

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/test.csv')

print(f'Train: {train.shape} | Test: {test.shape}')
train.head()

## 3. Exploratory Data Analysis

### 3.1 SalePrice Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('SalePrice Distribution')
axes[0].set_xlabel('SalePrice')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(train['SalePrice']), bins=50, color='tomato', edgecolor='white')
axes[1].set_title('Log SalePrice Distribution')
axes[1].set_xlabel('log(SalePrice)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'SalePrice  mean: {train["SalePrice"].mean():,.0f}')
print(f'SalePrice median: {train["SalePrice"].median():,.0f}')
print(f'SalePrice   skew: {train["SalePrice"].skew():.3f}')

### 3.2 Missing Values

In [ ]:
missing = pd.DataFrame({
    'Train Missing': train.isnull().sum(),
    'Train %':       (train.isnull().sum() / len(train) * 100).round(1),
    'Test Missing':  test.isnull().sum(),
    'Test %':        (test.isnull().sum() / len(test) * 100).round(1)
}).query('`Train Missing` > 0 or `Test Missing` > 0').sort_values('Train %', ascending=False)

print(missing.head(20))

### 3.3 Numeric Feature Correlations with SalePrice

In [ ]:
num_corr = (
    train.select_dtypes('number')
    .corr()['SalePrice']
    .drop('SalePrice')
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

num_corr.sort_values().plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.title('Top 10 Numeric Features by Correlation with SalePrice')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()

print(num_corr)

### 3.4 Key Categorical Features vs SalePrice

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# OverallQual vs SalePrice
train.groupby('OverallQual')['SalePrice'].median().plot(
    kind='bar', ax=axes[0], color='steelblue'
)
axes[0].set_title('Median SalePrice by Overall Quality')
axes[0].set_xlabel('OverallQual')
axes[0].set_ylabel('Median SalePrice')
axes[0].tick_params(axis='x', rotation=0)

# Neighborhood median SalePrice
nbhd = train.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=True)
nbhd.plot(kind='barh', ax=axes[1], color='tomato')
axes[1].set_title('Median SalePrice by Neighborhood')
axes[1].set_xlabel('Median SalePrice')

plt.tight_layout()
plt.show()

**EDA Findings:**
- SalePrice is right-skewed (skew ~1.88). Log transformation produces a near-normal distribution, which is why RMSLE is the right metric.
- OverallQual has the highest numeric correlation with SalePrice (~0.79). Quality ratings are the single best predictor.
- GrLivArea, GarageCars, GarageArea, and TotalBsmtSF are also strongly correlated (above 0.60).
- Several columns have very high missingness: PoolQC (99%), MiscFeature (96%), Alley (94%), Fence (81%), FireplaceQu (47%). These will be dropped.
- Neighborhood has strong price variation. NoRidge and NridgHt command the highest medians; MeadowV and BrDale the lowest.

## 4. Feature Engineering

### 4.1 Drop High-Missingness Columns

In [ ]:
HIGH_MISS = ['Alley', 'PoolQC', 'Fence', 'MiscFeature', 'FireplaceQu']

train = train.drop(columns=HIGH_MISS)
test  = test.drop(columns=HIGH_MISS)

print(f'Columns after dropping high-missingness: {train.shape[1]}')

### 4.2 Impute Missing Values

In [ ]:
for df in [train, test]:
    for col in df.select_dtypes('number').columns:
        df[col] = df[col].fillna(df[col].median())
    for col in df.select_dtypes('object').columns:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f'Train nulls remaining: {train.isnull().sum().sum()}')
print(f'Test  nulls remaining: {test.isnull().sum().sum()}')

### 4.3 Encode Ordinal Quality Columns

In [ ]:
QUAL_MAP = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
QUAL_COLS = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
             'HeatingQC', 'KitchenQual', 'GarageQual', 'GarageCond']

for df in [train, test]:
    for col in QUAL_COLS:
        if col in df.columns:
            df[col] = df[col].map(QUAL_MAP).fillna(0).astype(int)

print('Ordinal encoding complete.')

### 4.4 One-Hot Encode Remaining Categoricals

In [ ]:
cat_cols = train.select_dtypes('object').columns.tolist()

train = pd.get_dummies(train, columns=cat_cols)
test  = pd.get_dummies(test,  columns=cat_cols)

print(f'Train shape after encoding: {train.shape}')
print(f'Test  shape after encoding: {test.shape}')

### 4.5 Log-Transform Skewed Numeric Features

In [ ]:
SKEW_THRESHOLD = 0.75

# Use train to identify skewed columns (exclude target)
numeric_cols = train.select_dtypes('number').columns.drop('SalePrice', errors='ignore')
skewed = train[numeric_cols].apply(lambda x: x.skew()).abs()
skewed_cols = skewed[skewed > SKEW_THRESHOLD].index.tolist()

for df in [train, test]:
    for col in skewed_cols:
        if col in df.columns:
            df[col] = np.log1p(df[col].clip(lower=0))

print(f'Log-transformed {len(skewed_cols)} skewed features.')

### 4.6 Align Train and Test Columns

In [ ]:
TARGET = 'SalePrice'
FEATURES = [c for c in train.columns if c not in [TARGET, 'Id']]

# Keep only columns present in both after one-hot encoding
shared_cols = [c for c in FEATURES if c in test.columns]

X      = train[shared_cols]
y      = np.log1p(train[TARGET])
X_test = test[shared_cols]

print(f'Features: {len(shared_cols)} | Train rows: {len(X)} | Test rows: {len(X_test)}')

## 5. Model: RandomForestRegressor

In [ ]:
model = RandomForestRegressor(
    n_estimators      = 300,
    max_depth         = None,
    min_samples_split = 4,
    random_state      = SEED,
    n_jobs            = -1
)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
cv_rmsle  = -cv_scores

print(f'CV RMSLE: {cv_rmsle.mean():.4f} (std {cv_rmsle.std():.4f})')
print(f'Per-fold: {np.round(cv_rmsle, 4)}')

In [ ]:
model.fit(X, y)

train_preds   = model.predict(X)
train_rmsle   = np.sqrt(mean_squared_error(y, train_preds))
print(f'Train RMSLE (in-sample): {train_rmsle:.4f}')

## 6. Evaluation

In [ ]:
importances = pd.Series(model.feature_importances_, index=shared_cols).sort_values(ascending=False)

importances.head(20).sort_values().plot(kind='barh', figsize=(8, 7), color='steelblue')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(importances.head(10).round(4))

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y, train_preds, alpha=0.3, color='steelblue', s=10)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=1)
plt.title('Predicted vs Actual Log SalePrice (Train)')
plt.xlabel('Actual log(SalePrice)')
plt.ylabel('Predicted log(SalePrice)')
plt.tight_layout()
plt.show()

## 7. Generate Submission

In [ ]:
log_preds = model.predict(X_test)
preds     = np.expm1(log_preds)

submission = pd.DataFrame({
    'Id':        test['Id'],
    'SalePrice': preds
})

submission.to_csv('submission.csv', index=False)
print(f'Submission saved: {submission.shape[0]} rows')
print(f'Predicted SalePrice range: {preds.min():,.0f} to {preds.max():,.0f}')
submission.head()

## Summary
- **Model:** RandomForestRegressor (300 trees, min_samples_split=4)
- **Target:** log1p(SalePrice). Predictions exponentiated back to price scale.
- **Features:** ~200+ after one-hot encoding. Top predictors: OverallQual, GrLivArea, TotalBsmtSF, GarageArea.
- **CV RMSLE:** see cell output above
- **Limitations:** No hyperparameter tuning. Median/mode imputation may lose signal in structured missingness. No interaction features.
- **Next steps:** Try GradientBoostingRegressor or XGBoost. Add neighborhood price aggregates. Run RandomizedSearchCV for tuning.